# NQ Intraday Research Notebook

Notebook central pour piloter le pipeline data -> features -> stratégie ORB -> backtest -> diagnostics.

## 1) Setup projet

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config.paths import RAW_DATA_DIR, ensure_directories
from src.data.loader import load_ohlcv_csv
from src.data.validation import validate_ohlcv
from src.data.cleaning import clean_ohlcv
from src.data.session import extract_rth, add_session_date
from src.features.intraday import add_intraday_features
from src.features.opening_range import compute_opening_range
from src.features.returns import add_simple_returns, add_log_returns
from src.features.volatility import add_rolling_std, add_atr
from src.strategy.orb import ORBStrategy
from src.engine.execution_model import ExecutionModel
from src.engine.backtester import run_backtest
from src.engine.portfolio import build_equity_curve
from src.analytics.metrics import compute_metrics
from src.analytics.diagnostics import performance_by_weekday, performance_by_month, performance_by_year
from src.analytics.heatmaps import run_orb_grid_search, pivot_heatmap
from src.visualization.plots import plot_trade_histogram, plot_pnl_distribution, plot_cumulative_pnl
from src.visualization.equity import plot_equity_curve, plot_drawdown_curve

ensure_directories()
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

## 2) Chargement d’un sample 1-minute

In [ ]:
df_raw = load_ohlcv_csv(RAW_DATA_DIR / 'NQ_1min_sample.csv')
df_raw.head()

## 3) Contrôle qualité de la donnée

In [ ]:
quality_report = validate_ohlcv(df_raw)
quality_report

## 4) Nettoyage

In [ ]:
df = clean_ohlcv(df_raw)
len(df), df.head()

## 5) Filtrage de session (RTH)

In [ ]:
df_rth = extract_rth(df)
df_rth = add_session_date(df_rth)
df_rth[['timestamp', 'session_date']].head()

## 6) Construction de features intraday

In [ ]:
df_feat = add_intraday_features(df_rth)
df_feat = add_simple_returns(df_feat)
df_feat = add_log_returns(df_feat)
df_feat = add_rolling_std(df_feat, window=5)
df_feat = add_atr(df_feat, window=5)
df_feat.head()

## 7) Calcul de l’opening range

In [ ]:
df_or = compute_opening_range(df_feat, or_minutes=3)
df_or[['timestamp', 'session_date', 'or_high', 'or_low', 'or_width', 'or_midpoint']].head(10)

## 8) Backtest ORB baseline

In [ ]:
strategy = ORBStrategy(or_minutes=3, direction='both', one_trade_per_day=True, entry_buffer_ticks=0, stop_multiple=1.0, target_multiple=1.5, time_exit='15:55')
df_sig = strategy.generate_signals(df_or)
trades = run_backtest(df_sig, execution_model=ExecutionModel(), time_exit=strategy.time_exit, stop_multiple=strategy.stop_multiple, target_multiple=strategy.target_multiple)
trades

## 9) Analyse des trades

In [ ]:
metrics = compute_metrics(trades)
metrics

In [ ]:
performance_by_weekday(trades)

In [ ]:
performance_by_month(trades)

In [ ]:
performance_by_year(trades)

## 10) Equity curve

In [ ]:
equity = build_equity_curve(trades)
equity.head()

In [ ]:
if not trades.empty:
    plot_trade_histogram(trades)
    plot_pnl_distribution(trades)
    plot_cumulative_pnl(trades)
    plot_equity_curve(equity)
    plot_drawdown_curve(equity)
    plt.show()

## 11) Exemple de mini grid search

In [ ]:
grid = run_orb_grid_search(df_or, or_minutes_values=[2, 3, 4], target_multiple_values=[1.0, 1.5, 2.0])
grid

## 12) Heatmap simple

In [ ]:
heatmap_df = pivot_heatmap(grid, index='or_minutes', columns='target_multiple', values='cumulative_pnl')
plt.figure(figsize=(6, 4))
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='RdYlGn')
plt.title('Grid Search Heatmap - Cumulative PnL')
plt.tight_layout()
plt.show()

## 13) Next steps

- Remplacer les CSV samples par historique complet Bloomberg/futures.
- Ajouter découpage in-sample / out-of-sample et walk-forward.
- Ajouter stress tests coûts/slippage et filtres de volatilité.
- Préparer une couche broker/exécution live avec risk checks.